# PPO with a learned continuous reward-gap correction

This notebook calibrates the proxy/judge gap on the reference policy, trains a first policy, trains a `GapFinder` on that policy's normalized gaps, then uses the GapFinder as a generic reward adjustment for a second PPO run. The final policy is evaluated on a separate held-out prompt range.

Run the cells in order. The default run uses 1,000 examples and can take a long time. Reduce `RUN_END` for a smoke test.

In [ ]:
from dataclasses import replace
import logging
from pathlib import Path

import functions
from Datasets.dataset_request import RequestDataset
from Models.reward_adjustment import GapFinderCorrection
from functions import (
    ConfigTrainClassifier,
    DatasetSpec,
    EvaluateConfig,
    EvaluatorSpec,
    PolicySpec,
    RewardSpec,
    TrainingPPOConfig,
)

# Avoid extremely noisy HTTP debug logs while retaining training progress.
logging.getLogger().setLevel(logging.INFO)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
DATASET_NAME = "Anthropic/hh-rlhf"
RUN_START = 0
RUN_END = 1000
EVAL_START = 1000
EVAL_END = 1200

policy_spec = PolicySpec(
    model_name="Qwen/Qwen3-0.6B",
)
proxy_spec = RewardSpec(
    class_name="RewardModel",
    model_name="Skywork/Skywork-Reward-V2-Qwen3-0.6B",
    mode_name="proxy",
)
judge_spec = RewardSpec(
    class_name="RewardModel",
    model_name="Skywork/Skywork-Reward-V2-Qwen3-4B",
    mode_name="judge",
)

gap_config = ConfigTrainClassifier(
    dataset_name=DATASET_NAME,
    policy=policy_spec,
    reward=proxy_spec,
    judge=judge_spec,
    start_dataset=RUN_START,
    end_dataset=RUN_END,
    generation_batch_size=4,
    reward_batch_size=4,
    judge_batch_size=1,
    gap_finder_batch_size=4,
)

ppo_dataset = DatasetSpec(
    dataset_name=DATASET_NAME,
    start=RUN_START,
    end=RUN_END,
)

## 1. Calculate and save the frozen reference calibration

In [ ]:
gap_calibration = functions.calculate_gap_calibration(gap_config)
calibration_path = gap_calibration.save(
    "outputs/gap_finders/reference_calibration.json"
)
gap_calibration, calibration_path

## 2. Train the first policy with the normalized proxy reward

In [ ]:
normalized_proxy_spec = replace(
    proxy_spec,
    mean=gap_calibration.proxy_mean,
    std=gap_calibration.proxy_std,
)

first_ppo_config = TrainingPPOConfig(
    policy=policy_spec,
    reward=normalized_proxy_spec,
    dataset=ppo_dataset,
    output_dir="outputs/ppo_without_gap_finder",
    batch_size=1,
    gradient_accumulation_steps=8,
    rollout_forward_batch_size=1,
    generation_batch_size=2,
    response_length=64,
    logging_steps=1,
)

first_policy = functions.temp_ppo_train_policy(first_ppo_config)
first_policy_path = Path(first_ppo_config.output_dir) / "final"
first_policy_path

## 3. Train GapFinder on the first policy

The regression target is `proxy_z - judge_z`, using the frozen reference calibration above.

In [ ]:
gap_finder, reused_calibration = functions.create_gap_finder(
    gap_config,
    policy=first_policy,
    calibration=gap_calibration,
)

assert reused_calibration is gap_calibration
gap_finder.checkpoint_path

## 4. Attach GapFinder to the reward and train a second policy

`GapFinderCorrection` converts the predicted normalized gap into a raw proxy-score correction. The reward model only executes the generic adjustment.

In [ ]:
gap_adjustment = GapFinderCorrection(
    gap_finder,
    reward_std=gap_calibration.proxy_std,
)

second_policy_spec = replace(
    policy_spec,
    checkpoint=first_policy_path,
)
second_ppo_config = replace(
    first_ppo_config,
    policy=second_policy_spec,
    output_dir="outputs/ppo_with_gap_finder",
)

second_policy = functions.temp_ppo_train_policy(
    second_ppo_config,
    adjustments=(gap_adjustment,),
)
second_policy_path = Path(second_ppo_config.output_dir) / "final"
second_policy_path

## 5. Generate answers for held-out evaluation prompts

The PPO helper initially saves answers for its training range. This cell replaces that checkpoint dataset with prompts from `[EVAL_START, EVAL_END)` so the evaluation is held out.

In [ ]:
raw_evaluation_dataset = functions.load_dataset(DATASET_NAME)
evaluation_dataset = RequestDataset.from_raw(
    raw_evaluation_dataset,
    policy_spec.model_name,
    start=EVAL_START,
    end=EVAL_END,
)

second_policy.generate_new_dataset(evaluation_dataset, batch_size=2)
second_policy.save_dataset(second_policy_path)
second_policy.offload()
functions.empty_cuda_cache()
len(second_policy.dataset)

## 6. Evaluate the new policy

In [ ]:
evaluation_config = EvaluateConfig(
    policy=replace(policy_spec, checkpoint=second_policy_path),
    evaluator=EvaluatorSpec(
        class_name="PrometheusEvaluator",
        model_name="prometheus-eval/prometheus-7b-v2.0",
    ),
    evaluator_batch_size=1,
)

final_evaluation_score = functions.evaluate_policy(evaluation_config)
print(f"Final held-out evaluation score: {final_evaluation_score:.4f}")